In [3]:
import gymnasium
import highway_env
import time

# 1. Instantiate the environment
env = gymnasium.make(
    "racetrack-v0",
    render_mode="human",
    config={
        "action": {"type": "DiscreteMetaAction",
                   "acceleration_range": (-8.0, 5.0),
                   },
        "other_vehicles": 0,
        "controlled_vehicles": 1,
        "duration": 50,             # Increased duration to ensure enough time for a lap and stop
        "simulation_frequency": 25,  # Increased simulation frequency for better precision
        "policy_frequency": 25,      # Increased policy frequency for more responsive control
        'show_trajectories': True,
        "target_speeds": [0, 5, 10], # Align target speeds with desired behavior
    }
)

obs, info = env.reset()

# Force initial state
vehicle = env.unwrapped.vehicle
vehicle.speed = 0.0
vehicle.target_speed = 0.0

laps = 0
previous_node = vehicle.lane_index[0]
lap_completed_for_braking = False # New flag to ensure braking only after first lap


# Define control phases: 'ACCELERATING', 'CRUISING', 'BRAKING', 'FINISHED'
phase = 'ACCELERATING'

print("Simulation started. Phase: ACCELERATING")

while phase != 'FINISHED':
    current_node = vehicle.lane_index[0]
    current_speed = vehicle.speed
    
    # --- LAP TRACKING LOGIC ---
    if previous_node == "i" and current_node == "a":
        laps += 1
        print(f"Lap boundary crossed! Total laps: {laps}")
        if laps == 1: # Set flag when the first lap is completed
            lap_completed_for_braking = True
    
    # --- PHASE & SPEED CONTROL LOGIC ---
    
    # Phase 1: Accelerate up to ~10 m/s
    if phase == 'ACCELERATING':
        vehicle.target_speed = 10.0
        if current_speed < 9.5:
            action = env.unwrapped.action_type.actions_indexes["FASTER"]
        else:
            phase = 'CRUISING'
            print("Target speed reached. Phase: CRUISING")
            action = env.unwrapped.action_type.actions_indexes["IDLE"]

    # Phase 2: Stay around 10 m/s until nearing the end of the lap
    elif phase == 'CRUISING':
        vehicle.target_speed = 10.0
        
        # We start braking early on node 'i' *only if* the first lap is completed
        if lap_completed_for_braking and current_node == "h": # Changed from "i" to "h"
            phase = 'BRAKING'
            print("Approaching finish line. Phase: BRAKING")
            action = env.unwrapped.action_type.actions_indexes["SLOWER"]
        else:
            # Maintain speed
            if current_speed < 9.5:
                action = env.unwrapped.action_type.actions_indexes["FASTER"]
            elif current_speed > 10.5:
                action = env.unwrapped.action_type.actions_indexes["SLOWER"]
            else:
                action = env.unwrapped.action_type.actions_indexes["IDLE"]

    # Phase 3: Bring the vehicle to a complete stop on the line
    elif phase == 'BRAKING':
        vehicle.target_speed = 0.0
        
        if current_speed > 0.2:
            action = env.unwrapped.action_type.actions_indexes["SLOWER"]
        else:
            action = env.unwrapped.action_type.actions_indexes["IDLE"]
            # Stop condition: speed is effectively 0 and we are at the start node 'a'
            # The lap_completed_for_braking flag ensures we are doing this after the first lap.
            if current_node == "a" and current_speed <= 0.05: # Changed OR to AND
                phase = 'FINISHED'
                print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")

    # Execute action
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
    previous_node = current_node

    if done or truncated:
        print("Vehicle crashed or execution timed out.")
        break

print("Simulation finished successfully.")
env.close()

Simulation started. Phase: ACCELERATING
Target speed reached. Phase: CRUISING
Lap boundary crossed! Total laps: 1
Vehicle crashed or execution timed out.
Simulation finished successfully.


In [ ]:
import gymnasium
import highway_env
import time

# 1. Instantiate the environment
env = gymnasium.make(
    "racetrack-v0",
    render_mode="human",
    config={
        "action": {"type": "DiscreteMetaAction",
                   #"acceleration_range": (-8.0, 2.5),
                   },
        "other_vehicles": 0,
        "controlled_vehicles": 1,
        "duration": 100,             # Increased duration to ensure enough time for a lap and stop
        "simulation_frequency": 25,  # Increased simulation frequency for better precision
        "policy_frequency": 25,      # Increased policy frequency for more responsive control
        'show_trajectories': True,
        #"target_speeds": [0, 5, 10], # Align target speeds with desired behavior
    }
)

obs, info = env.reset()

# Force initial state
vehicle = env.unwrapped.vehicle
vehicle.speed = 0.0
vehicle.target_speed = 0.0

laps = 0
previous_node = vehicle.lane_index[0]
lap_completed_for_braking = False # New flag to ensure braking only after first lap


# Define control phases: 'ACCELERATING', 'CRUISING', 'BRAKING', 'FINISHED'
phase = 'ACCELERATING'

print("Simulation started. Phase: ACCELERATING")

while phase != 'FINISHED':
    current_node = vehicle.lane_index[0]
    current_speed = vehicle.speed
    print('phase:',phase,'current_speed:',current_speed)
    # --- LAP TRACKING LOGIC ---
    if previous_node == "i" and current_node == "a":
        laps += 1
        print(f"Lap boundary crossed! Total laps: {laps}")
        if laps == 1: # Set flag when the first lap is completed
            lap_completed_for_braking = True
    
    # --- PHASE & SPEED CONTROL LOGIC ---
    
    # Phase 1: Accelerate up to ~10 m/s
    if phase == 'ACCELERATING':
        vehicle.target_speed = 10.0
        if current_speed < 9.5:
            action = env.unwrapped.action_type.actions_indexes["FASTER"]
        else:
            phase = 'CRUISING'
            print("Target speed reached. Phase: CRUISING")
            action = env.unwrapped.action_type.actions_indexes["IDLE"]

    # Phase 2: Stay around 10 m/s until nearing the end of the lap
    elif phase == 'CRUISING':
        vehicle.target_speed = 10.0
        
        # We start braking early on node 'i' *only if* the first lap is completed
        #if lap_completed_for_braking and current_node == "h": # Changed from "i" to "h"
        if laps >= 1:
            phase = 'BRAKING'
            print("Approaching finish line. Phase: BRAKING")
            action = env.unwrapped.action_type.actions_indexes["SLOWER"]
        else:
            # Maintain speed
            if current_speed < 9.5:
                action = env.unwrapped.action_type.actions_indexes["FASTER"]
            elif current_speed > 10.5:
                action = env.unwrapped.action_type.actions_indexes["SLOWER"]
            else:
                action = env.unwrapped.action_type.actions_indexes["IDLE"]

    # Phase 3: Bring the vehicle to a complete stop on the line
    elif phase == 'BRAKING':
        vehicle.target_speed = 0.0
        
        if current_speed > 0.2:
            action = env.unwrapped.action_type.actions_indexes["SLOWER"]
        else:
            action = env.unwrapped.action_type.actions_indexes["IDLE"]import gymnasium
            import highway_env
            import time
            
            # 1. Instantiate the environment
            env = gymnasium.make(
                "racetrack-v0",
                render_mode="human",
                config={
                    "action": {
                        "type": "DiscreteMetaAction",
                        "target_speeds": [0, 5, 10], # Correctly nested
                    },
                    "other_vehicles": 0,
                    "controlled_vehicles": 1,
                    "duration": 100,
                    "simulation_frequency": 25,
                    "policy_frequency": 25,
                    'show_trajectories': True,
                }
            )
            
            obs, info = env.reset()
            
            # Use the Action Type's built-in helper to find indices
            actions = env.unwrapped.action_type.actions_indexes
            vehicle = env.unwrapped.vehicle
            
            laps = 0
            previous_node = vehicle.lane_index[0]
            phase = 'ACCELERATING'
            
            print("Simulation started. Phase: ACCELERATING")
            
            while phase != 'FINISHED':
                current_node = vehicle.lane_index[0]
                current_speed = vehicle.speed
                
                # --- LAP TRACKING LOGIC ---
                if previous_node == "i" and current_node == "a":
                    laps += 1
                    print(f"Lap boundary crossed! Total laps: {laps}")
                
                # --- PHASE & SPEED CONTROL LOGIC ---
                
                # Note: We no longer manually set vehicle.target_speed.
                # We let the environment's action handler manage it via FASTER/SLOWER.
            
                if phase == 'ACCELERATING':
                    if current_speed < 9.5:
                        action = actions["FASTER"]
                    else:
                        phase = 'CRUISING'
                        print("Target speed reached. Phase: CRUISING")
                        action = actions["IDLE"]
            
                elif phase == 'CRUISING':
                    if laps >= 1:
                        phase = 'BRAKING'
                        print("Approaching finish line. Phase: BRAKING")
                        action = actions["SLOWER"]
                    else:
                        # Simple cruise maintenance
                        if current_speed < 9.0:
                            action = actions["FASTER"]
                        elif current_speed > 11.0:
                            action = actions["SLOWER"]
                        else:
                            action = actions["IDLE"]
            
                elif phase == 'BRAKING':
                    if current_speed > 0.1:
                        action = actions["SLOWER"]
                    else:
                        action = actions["IDLE"]
                        # Ensure we are at the finish line node 'a' and fully stopped
                        if current_node == "a" and current_speed <= 0.05:
                            phase = 'FINISHED'
                            print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")
            
                # Execute action
                obs, reward, done, truncated, info = env.step(action)
                env.render()
                
                previous_node = current_node
            
                if done or truncated:
                    print("Vehicle crashed or execution timed out.")
                    break
            
            print("Simulation finished successfully.")
            env.close()
            
            # Stop condition: speed is effectively 0 and we are at the start node 'a'
            # The lap_completed_for_braking flag ensures we are doing this after the first lap.
            if current_node == "a" and current_speed <= 0.05: # Changed OR to AND
                phase = 'FINISHED'
                print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")

    # Execute action
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
    previous_node = current_node

    if done or truncated:
        print("Vehicle crashed or execution timed out.")
        break

print("Simulation finished successfully.")
env.close()

SyntaxError: invalid syntax (698103778.py, line 90)

: 